# Clase 7: pandas en Profundidad — Del Dato al Análisis

**Diplomado en Data Science Aplicada con Python para la Toma de Decisiones**  
Arca Continental Ecuador · UDLA · 2026

---

**Instructor:** Carlos Enrique Mosquera Trujillo  
**Contacto:** cmosquerat@unal.edu.co

---

### Contenido de esta sesión

| # | Tema | Duración |
|---|------|----------|
| 1 | Cargar un dataset desde GitHub | 5 min |
| 2 | Selección de datos: columnas, `loc`, `iloc` | 10 min |
| 3 | Explorar valores antes de filtrar | 5 min |
| 4 | Filtrado: máscaras booleanas, `isin`, `between` | 15 min |
| 5 | Transformar: columnas derivadas y `apply()` | 15 min |
| 6 | Agrupar y resumir: `groupby()` + `agg()` | 15 min |
| 7 | Datos faltantes: NaN | 15 min |
| 8 | Visualización con `.plot()` | 10 min |
| 9 | Proyecto integrador | 15 min |

---
## 1. Cargar un Dataset desde GitHub

`pd.read_csv()` acepta URLs — no necesitas descargar el archivo manualmente.

Usaremos el dataset **Titanic**, un estándar en Data Science. Tiene números, texto, valores faltantes y una pregunta analítica clara: **¿quién sobrevivió y por qué?**

| Columna | Descripción |
|---------|-------------|
| `Survived` | 0=No, 1=Sí |
| `Pclass` | Clase (1=1ra, 2=2da, 3=3ra) |
| `Sex` | male / female |
| `Age` | Edad (con valores faltantes) |
| `SibSp` | Hermanos/cónyuge a bordo |
| `Parch` | Padres/hijos a bordo |
| `Fare` | Tarifa pagada |
| `Embarked` | Puerto (S/C/Q) |

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

print(f"Filas: {df.shape[0]}, Columnas: {df.shape[1]}")
print(f"Columnas: {list(df.columns)}")
df.head()

---
## 2. Selección de Datos

### Seleccionar columnas

| Sintaxis | Resultado |
|----------|----------|
| `df["Age"]` | Series (una columna) |
| `df[["Age"]]` | DataFrame de 1 columna |
| `df[["Age", "Sex"]]` | DataFrame de 2 columnas |
| `df.Age` | Series (acceso por atributo) |

In [ ]:
# Una columna → Series
edades = df["Age"]
print("Tipo:", type(edades).__name__)
print(edades.head())

print()

# Varias columnas → DataFrame
subdf = df[["Sex", "Age", "Survived"]]
print("Tipo:", type(subdf).__name__)
subdf.head()

### Acceso preciso: `loc` e `iloc`

- **`.loc[]`** — por etiqueta (nombre de columna) o condición.
- **`.iloc[]`** — por posición numérica (índice entero).

El 90% de las veces usarás `loc`. Solo necesitas `iloc` cuando no conoces los nombres de columna.

In [ ]:
# loc — por nombre
print("Fila 0, columna Sex:", df.loc[0, "Sex"])
print()
print(df.loc[0:3, ["Sex", "Age", "Survived"]])

In [ ]:
# iloc — por posición
print("Fila 0, columna 2:", df.iloc[0, 2])
print()
print(df.iloc[0:4, [2, 3, 5]])

---
## 3. Explorar Valores Antes de Filtrar

Antes de filtrar, necesitas saber **qué valores existen** en cada columna.

- `.unique()` → lista de valores distintos.
- `.value_counts()` → frecuencia de cada valor.
- `.min()` / `.max()` → rango.

In [ ]:
print("Sex — valores únicos:", df["Sex"].unique())
print()
print("Sex — conteo:")
print(df["Sex"].value_counts())

print()
print("Pclass — valores únicos:", df["Pclass"].unique())
print("Pclass — conteo:")
print(df["Pclass"].value_counts().sort_index())

print()
print(f"Age — mínimo: {df['Age'].min()}, máximo: {df['Age'].max()}")
print(f"Age — cuántos NaN: {df['Age'].isna().sum()}")

---
## 4. Filtrado de Datos

### Máscaras booleanas

Una condición sobre una columna produce una **Serie de True/False**. Al pasarla como índice, pandas devuelve solo las filas donde es `True`.

In [ ]:
# La máscara: columna de True/False
mascara = df["Sex"] == "female"
print("Máscara (primeros 8):", list(mascara[:8]))

# Aplicar la máscara para filtrar
mujeres = df[mascara]
print(f"\nMujeres: {len(mujeres)} de {len(df)} pasajeros")

# En una línea (lo más común)
primera_clase = df[df["Pclass"] == 1]
print(f"1ra clase: {len(primera_clase)} pasajeros")

# Filtrar + seleccionar columnas
jovenes = df[df["Age"] < 10][["Sex", "Age", "Pclass", "Survived"]]
print(f"\nMenores de 10 años ({len(jovenes)}):")
jovenes.head()

### Múltiples condiciones: `&`, `|`, `~`

| Operador | Significado | Equivalente en Python |
|----------|-------------|----------------------|
| `&` | Y lógico | `and` |
| `\|` | O lógico | `or` |
| `~` | NO lógico | `not` |

**Cada condición debe ir entre paréntesis.** Sin paréntesis, Python aplica los operadores en el orden incorrecto.

In [ ]:
# & = Y (ambas verdaderas)
mujeres_1ra = df[(df["Sex"] == "female") & (df["Pclass"] == 1)]
print(f"Mujeres en 1ra clase: {len(mujeres_1ra)}")

# | = O (al menos una verdadera)
ninos_o_1ra = df[(df["Age"] < 12) | (df["Pclass"] == 1)]
print(f"Niños O 1ra clase: {len(ninos_o_1ra)}")

# ~ = NO (invierte)
no_southampton = df[~(df["Embarked"] == "S")]
print(f"NO embarcaron en S: {len(no_southampton)}")

# Combinación
analisis = df[(df["Pclass"] == 3) & (df["Survived"] == 1)]
print(f"\nSupervivientes 3ra clase: {len(analisis)}")
print(f"Tasa: {len(analisis) / len(df[df['Pclass'] == 3]) * 100:.1f}%")

### Filtros especiales: `isin()` y `between()`

- `isin(lista)` — como `IN` de SQL. Evita cadenas de `|`.
- `between(a, b)` — rango numérico inclusivo.

In [ ]:
# isin()
no_southampton = df[df["Embarked"].isin(["C", "Q"])]
print(f"Embarcaron en C o Q: {len(no_southampton)}")

# between()
adultos_medios = df[df["Age"].between(30, 50)]
print(f"Edad 30-50: {len(adultos_medios)} pasajeros")

# Negación con isin
solo_southampton = df[~df["Embarked"].isin(["C", "Q"])]
print(f"Solo Southampton: {len(solo_southampton)}")

### Práctica: Filtrado (7 min)

Responde cada pregunta con un filtro. Imprime `len(resultado)` para verificar.

In [ ]:
# 1. ¿Cuántos pasajeros pagaron más de $50?
#    ¿Qué % del total representan?

# 2. Niños (< 10 años) que sobrevivieron

# 3. Pasajeros de 1ra o 2da clase (usa isin)

# 4. Adultos jóvenes (20-30 años) en 3ra clase
#    ¿Cuál fue su tasa de supervivencia?

In [ ]:
# Solución

# 1
premium = df[df["Fare"] > 50]
print(f"Fare > $50: {len(premium)} ({len(premium)/len(df)*100:.1f}%)")

# 2
ninos_vivos = df[(df["Age"] < 10) & (df["Survived"] == 1)]
print(f"Niños sobrevivientes: {len(ninos_vivos)}")

# 3
clase_1_2 = df[df["Pclass"].isin([1, 2])]
print(f"1ra o 2da clase: {len(clase_1_2)}")

# 4
jovenes_3ra = df[(df["Age"].between(20, 30)) & (df["Pclass"] == 3)]
tasa = jovenes_3ra["Survived"].mean() * 100
print(f"Jóvenes 20-30 en 3ra: {len(jovenes_3ra)}, supervivencia: {tasa:.1f}%")

---
## 5. Transformar Datos

### Crear columnas nuevas

Las operaciones en columnas son **vectorizadas**: aplican a toda la columna de una vez, mucho más rápido que un `for`.

In [ ]:
# Operación directa (vectorizada)
df["Fare_doble"] = df["Fare"] * 2

# Condición booleana como columna
df["Es_mayor"] = df["Age"] >= 18

# Categoría basada en condición
df["Rango_edad"] = "Desconocido"
df.loc[df["Age"] < 12, "Rango_edad"] = "Niño"
df.loc[(df["Age"] >= 12) & (df["Age"] < 18), "Rango_edad"] = "Adolescente"
df.loc[df["Age"] >= 18, "Rango_edad"] = "Adulto"

df[["Age", "Es_mayor", "Rango_edad", "Fare", "Fare_doble"]].head(10)

### `apply()` — Aplica una función a cada valor

`df["col"].apply(mi_funcion)` toma cada valor de la columna, lo pasa por tu función, y devuelve una columna nueva con los resultados.

Úsalo cuando necesitas **lógica con if/elif** que no cabe en una sola expresión.

In [ ]:
def etiqueta_clase(numero):
    if numero == 1:
        return "1ra Clase"
    elif numero == 2:
        return "2da Clase"
    else:
        return "3ra Clase"

df["Clase_texto"] = df["Pclass"].apply(etiqueta_clase)
df[["Pclass", "Clase_texto"]].head(10)

### Práctica: Columnas derivadas (7 min)

Enriquece el dataset con columnas calculadas.

In [ ]:
# Recarga el df limpio
df = pd.read_csv(url)

# 1. "Tarifa_alta" → True si Fare > 50

# 2. "Sobrevivio" → "Sí" o "No"  (usa apply o map)

# 3. "Familia" = SibSp + Parch

# 4. "Viaja_solo" → True si Familia == 0
#    ¿Sobrevivieron más los que viajaban solos?

df.head()

In [ ]:
# Solución

df = pd.read_csv(url)

df["Tarifa_alta"] = df["Fare"] > 50

df["Sobrevivio"] = df["Survived"].apply(lambda x: "Sí" if x == 1 else "No")

df["Familia"] = df["SibSp"] + df["Parch"]

df["Viaja_solo"] = df["Familia"] == 0

print(df[["Tarifa_alta", "Sobrevivio", "Familia", "Viaja_solo"]].head(8))

print("\nSupervivencia solos vs acompañados:")
print(df.groupby("Viaja_solo")["Survived"].mean().round(3))

---
## 6. Agrupar y Resumir: `groupby()`

Para responder preguntas del tipo **"¿cuánto/cuál es el promedio… por grupo?"**

### Anatomía de groupby

```python
df.groupby("Sex")["Age"].mean()
#  ①          ②     ③     ④
#  ① DataFrame
#  ② Agrupar por esta columna
#  ③ Calcular sobre esta columna
#  ④ Con esta función
```

| Función | Responde… |
|---------|-----------|
| `.count()` | ¿Cuántos hay? |
| `.sum()` | ¿Cuánto en total? |
| `.mean()` | ¿Cuál es el promedio? |
| `.min()` / `.max()` | ¿Cuál es el mínimo/máximo? |

In [ ]:
df = pd.read_csv(url)

# ¿Cuántos pasajeros hay por sexo?
print(df.groupby("Sex")["PassengerId"].count())

print()
# ¿Tarifa promedio por clase?
print(df.groupby("Pclass")["Fare"].mean().round(2))

print()
# ¿Cuántos sobrevivieron por clase?
print(df.groupby("Pclass")["Survived"].sum())

### `groupby()` + `agg()` — Varias métricas a la vez

En lugar de hacer tres `groupby` separados, `.agg()` calcula todo de una vez:

```python
df.groupby("Pclass").agg(
    total       = ("PassengerId", "count"),  # nombre = (columna, función)
    tarifa_prom = ("Fare",        "mean"),
)
```

In [ ]:
resumen = df.groupby("Pclass").agg(
    total       = ("PassengerId", "count"),
    sobreviven  = ("Survived",    "sum"),
    tarifa_prom = ("Fare",        "mean"),
).round(1)

print(resumen)

### Práctica: Agrupar y resumir (8 min)

In [ ]:
df = pd.read_csv(url)

# 1. Promedio de edad por sexo

# 2. Total de sobrevivientes por clase (Pclass)

# 3. Tabla resumen por clase: cantidad, suma de Fare,
#    promedio de Fare → usa .agg()

# RETO: ¿Qué puerto (Embarked) tuvo la mejor tasa
#        de supervivencia?

In [ ]:
# Solución

df = pd.read_csv(url)

# 1
print("Promedio de edad por sexo:")
print(df.groupby("Sex")["Age"].mean().round(1))

# 2
print("\nSobrevivientes por clase:")
print(df.groupby("Pclass")["Survived"].sum())

# 3
print("\nResumen por clase:")
resumen = df.groupby("Pclass").agg(
    cantidad    = ("PassengerId", "count"),
    fare_total  = ("Fare", "sum"),
    fare_prom   = ("Fare", "mean"),
).round(1)
print(resumen)

# RETO
print("\nSupervivencia por puerto:")
print(df.groupby("Embarked")["Survived"].mean().sort_values(ascending=False).round(3))

---
## 7. Datos Faltantes: NaN

### Detectar valores faltantes

- `.isna()` → máscara True/False.
- `.isna().sum()` → cuenta de NaN por columna.
- Un NaN **no es cero** — incluirlo en promedios distorsiona el análisis.

In [ ]:
df = pd.read_csv(url)

print("=== NaN por columna ===")
print(df.isna().sum())

print("\n=== % faltante ===")
nan_pct = (df.isna().sum() / len(df) * 100).round(1)
print(nan_pct[nan_pct > 0])

print(f"\nFilas con algún NaN: {df[df.isna().any(axis=1)].shape[0]}")

### Estrategias para limpiar NaN

| Estrategia | Cuándo usarla |
|------------|---------------|
| `dropna()` | Pocos NaN (< 5%), perder filas no importa |
| `fillna(mediana)` | Columnas numéricas — no distorsiona la distribución |
| `fillna("valor")` | Columnas de texto — el más frecuente o "Desconocido" |
| `fillna(0)` | Casi nunca — el 0 tiene significado propio |

In [ ]:
print(f"NaN en Age: {df['Age'].isna().sum()}")
print(f"NaN en Embarked: {df['Embarked'].isna().sum()}")

# Numérica → fillna con mediana
mediana = df["Age"].median()
df["Age"] = df["Age"].fillna(mediana)
print(f"\nAge después: {df['Age'].isna().sum()} NaN (mediana={mediana})")

# Categórica → fillna con valor más frecuente
df["Embarked"] = df["Embarked"].fillna("S")
print(f"Embarked después: {df['Embarked'].isna().sum()} NaN")

# dropna → eliminar filas donde Fare sea NaN
df2 = df.dropna(subset=["Fare"])
print(f"\ndropna(Fare): {len(df)} → {len(df2)} filas")

### Práctica: Limpiar valores faltantes (7 min)

Audita los NaN del dataset, decide cómo tratarlos, y compara el impacto.

In [ ]:
df = pd.read_csv(url)

# 1. ¿Cuántos NaN por columna? ¿Qué % representan?

# 2. Si haces dropna(), ¿cuántas filas pierdes?

# 3. Rellena Age con la mediana.
#    Compara df["Age"].mean() ANTES y DESPUÉS.

# 4. Rellena Embarked con el valor más frecuente.
#    Pista: df["Embarked"].mode()[0]

In [ ]:
# Solución

df = pd.read_csv(url)

# 1
print("NaN por columna:")
print(df.isna().sum())
pct = (df.isna().sum() / len(df) * 100).round(1)
print("\n% faltante:")
print(pct[pct > 0])

# 2
df_sin_nan = df.dropna()
print(f"\nFilas originales: {len(df)}, tras dropna: {len(df_sin_nan)} (pierdes {len(df)-len(df_sin_nan)})")

# 3
print(f"\nMedia ANTES: {df['Age'].mean():.2f}")
mediana = df["Age"].median()
df["Age"] = df["Age"].fillna(mediana)
print(f"Media DESPUÉS: {df['Age'].mean():.2f} (mediana usada: {mediana})")
print(f"NaN en Age: {df['Age'].isna().sum()}")

# 4
moda = df["Embarked"].mode()[0]
df["Embarked"] = df["Embarked"].fillna(moda)
print(f"\nEmbarked rellenado con '{moda}':")
print(df["Embarked"].value_counts())

---
## 8. Visualización con `.plot()`

Cualquier Serie o DataFrame tiene un método `.plot()` que genera una gráfica directamente.

| `kind=` | Cuándo usarlo |
|---------|---------------|
| `"bar"` | Comparar categorías |
| `"barh"` | Barras horizontales |
| `"hist"` | Distribución de valores |
| `"line"` | Tendencia en el tiempo |
| `"scatter"` | Relación entre dos columnas |

In [ ]:
import matplotlib.pyplot as plt

df = pd.read_csv(url)

# Supervivencia por clase
por_clase = df.groupby("Pclass")["Survived"].mean() * 100
por_clase.plot(
    kind="bar",
    color=["#16A34A", "#2563EB", "#C82B40"],
    title="Tasa de supervivencia por clase (%)"
)
plt.ylabel("%")
plt.tight_layout()
plt.show()

In [ ]:
# Supervivencia por sexo
por_sexo = df.groupby("Sex")["Survived"].mean() * 100
por_sexo.plot(kind="bar", title="Supervivencia por Sexo (%)",
              color=["#C82B40", "#2563EB"])
plt.ylabel("%")
plt.tight_layout()
plt.show()

In [ ]:
# Distribución de edades
df["Age"].dropna().plot(kind="hist", bins=20,
    color="#2563EB", edgecolor="white",
    title="Distribución de Edades")
plt.xlabel("Edad")
plt.tight_layout()
plt.show()

---
## 9. Proyecto Integrador: Análisis de Supervivencia

Realiza un análisis completo del dataset Titanic:

1. **Limpiar NaN** — rellena `Age` con mediana.
2. **Crear `"Grupo_edad"`** — Niño (< 15) / Adulto (15-60) / Mayor (> 60).
3. **Crear `"Familia"`** = SibSp + Parch.
4. **Tasa de supervivencia** por `Pclass` y `Sex`.
5. **¿Qué grupo tuvo la mayor/menor supervivencia?**
6. **(BONUS)** Graficar el resultado.

In [ ]:
import pandas as pd

df = pd.read_csv(url)

# PASO 1: Limpiar NaN
df["Age"] = df["Age"].fillna(df["Age"].median())

# PASO 2: Crear "Grupo_edad" (Niño/Adulto/Mayor)

# PASO 3: Crear "Familia" y "Viaja_solo"

# PASO 4: Supervivencia por Pclass × Sex

# PASO 5: ¿Qué grupo sobrevivió más/menos?

# BONUS: Gráfica

df.head()

In [ ]:
# Solución

import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(url)

# 1. Limpiar
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna("S")

# 2. Grupo_edad
def grupo_edad(age):
    if age < 15:
        return "Niño"
    elif age <= 60:
        return "Adulto"
    else:
        return "Mayor"

df["Grupo_edad"] = df["Age"].apply(grupo_edad)

# 3. Familia
df["Familia"] = df["SibSp"] + df["Parch"]
df["Viaja_solo"] = df["Familia"] == 0

# 4. Supervivencia por Pclass × Sex
print("=== Supervivencia por Clase y Sexo ===")
cruzado = df.groupby(["Pclass", "Sex"])["Survived"].mean().round(3) * 100
print(cruzado)

# 5. Insight
print("\n=== Supervivencia por Grupo de edad ===")
print(df.groupby("Grupo_edad")["Survived"].mean().round(3) * 100)

print("\n=== Solos vs Acompañados ===")
print(df.groupby("Viaja_solo")["Survived"].mean().round(3) * 100)

# BONUS: Gráfica
por_clase = df.groupby("Pclass")["Survived"].mean() * 100
por_clase.plot(kind="bar",
    color=["#16A34A", "#2563EB", "#C82B40"],
    title="Tasa de supervivencia por clase (%)")
plt.ylabel("%")
plt.tight_layout()
plt.show()

---
## Resumen: Lo que aprendimos hoy

- Cargar CSV desde URL con `pd.read_csv()`.
- **Seleccionar**: `df["col"]`, `df[["a","b"]]`, `loc[]`, `iloc[]`.
- **Explorar**: `.unique()`, `.value_counts()`.
- **Filtrar**: máscaras booleanas `&`, `|`, `~`, `isin()`, `between()`.
- **Transformar**: columnas derivadas, `apply()`.
- **Agrupar**: `groupby()` + `agg()`.
- **Limpiar NaN**: `isna()`, `fillna()`, `dropna()`.
- **Visualizar**: `.plot()` con matplotlib.

### Próxima clase

- **Análisis Exploratorio de Datos (EDA)** — entender un dataset antes de modelar.
- **Estadística descriptiva**: media, mediana, distribución, outliers.
- **Correlaciones** y relaciones entre variables.